In [13]:
import json
import pickle
from shapely.geometry import Polygon
import networkx as nx
from networkx.readwrite import json_graph

In [3]:

graph_pkl = "./output/graphs_reoriented.pkl"
# Load graphs
with open(graph_pkl, 'rb') as f:
    graphs = pickle.load(f)

In [4]:
graphs[0]

{'filename': '40795.png',
 'rooms': [{'type': 'common_room',
   'polygon': <POLYGON ((69 80, 69 121, 117 121, 117 80, 69 80))>,
   'area': 1968.0,
   'centroid': (93.0, 100.5)},
  {'type': 'common_room',
   'polygon': <POLYGON ((23 100, 23 134, 65 134, 65 100, 23 100))>,
   'area': 1428.0,
   'centroid': (44.0, 117.0)},
  {'type': 'master_room',
   'polygon': <POLYGON ((152 80, 152 121, 214 121, 214 80, 152 80))>,
   'area': 2542.0,
   'centroid': (183.0, 100.5)},
  {'type': 'living_room',
   'polygon': <POLYGON ((121 107, 121 124, 120 125, 69 125, 69 159, 120 159, 121 160, 121 ...>,
   'area': 6998.5,
   'centroid': (146.88495153723417, 145.95382343835584)},
  {'type': 'balcony',
   'polygon': <POLYGON ((218 80, 218 121, 233 121, 233 80, 218 80))>,
   'area': 615.0,
   'centroid': (225.5, 100.5)},
  {'type': 'bathroom',
   'polygon': <POLYGON ((121 80, 121 103, 148 103, 148 80, 121 80))>,
   'area': 621.0,
   'centroid': (134.5, 91.5)},
  {'type': 'kitchen',
   'polygon': <POLYGON ((2

In [14]:
def serialize_polygon_wkt(poly: Polygon) -> str:
    """Convert a Shapely Polygon to WKT."""
    return poly.wkt

def serialize_polygon_geojson(poly: Polygon) -> str:
    """Convert Polygon to GeoJSON string."""
    return json.dumps(poly.__geo_interface__)

def serialize_point(pt) :
    # or pt.wkt, but tuple is often more convenient downstream
    return (pt.x, pt.y)

def serialize_graph(G: nx.Graph) -> dict:
    """
    Convert a NetworkX Graph into node-link JSON format.
    - 'nodes': list of {id: node_id, **node_attributes}
    - 'links': list of {source: node_id, target: node_id, **edge_attributes}
    """
    return json_graph.node_link_data(G)

In [25]:
out_dict = {}

In [26]:
for graph in graphs:
    filename = graph['filename'].split(".")[0]
    rooms = graph['rooms']
    G = graph['graph']

    out_dict[filename] = {
                                'rooms': [
                                    {
                                        'type': room['type'],
                                        'polygon': serialize_polygon_wkt(room['polygon']),
                                        # 'polygon': serialize_polygon_geojson(room['polygon']),
                                        'area': room['area'],
                                        'centroid': room['centroid'],
                                    }
                                    for room in rooms
                                ],
                                'graph': serialize_graph(G)
                            }
    

In [22]:
out_dict['40795']

{'rooms': [{'type': 'common_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[69.0, 80.0], [69.0, 121.0], [117.0, 121.0], [117.0, 80.0], [69.0, 80.0]]]}',
   'area': 1968.0,
   'centroid': (93.0, 100.5)},
  {'type': 'common_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[23.0, 100.0], [23.0, 134.0], [65.0, 134.0], [65.0, 100.0], [23.0, 100.0]]]}',
   'area': 1428.0,
   'centroid': (44.0, 117.0)},
  {'type': 'master_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[152.0, 80.0], [152.0, 121.0], [214.0, 121.0], [214.0, 80.0], [152.0, 80.0]]]}',
   'area': 2542.0,
   'centroid': (183.0, 100.5)},
  {'type': 'living_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[121.0, 107.0], [121.0, 124.0], [120.0, 125.0], [69.0, 125.0], [69.0, 159.0], [120.0, 159.0], [121.0, 160.0], [121.0, 176.0], [214.0, 176.0], [214.0, 125.0], [149.0, 125.0], [148.0, 124.0], [148.0, 107.0], [121.0, 107.0]]]}',
   'area': 6998.5,
   'centroid': (146.88495153723417, 145.95382

In [27]:
with open('graphs_reoriented_dict.pkl', 'wb') as f:
    pickle.dump(out_dict, f)

In [23]:
import json 
with open('graph_dict.json', 'w') as f:
    json.dump(out_dict, f, indent=2)

TypeError: Object of type Polygon is not JSON serializable